# Chapter 6 Lab: Designing the Dataset (`ch04b`)

In this lab, we explore practical dataset design challenges: label noise, class imbalance, augmentation validity, and active learning. Each section builds intuition about how to detect and handle common data quality issues.

## Setup

This notebook uses scikit-learn for classic ML models and metrics, and matplotlib for visualization. All code is self-contained and reproducible with fixed random seeds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

%matplotlib inline
np.random.seed(42)

## Part 1: Label Noise and Model Robustness

When labels are mislabeled (due to annotator error or data entry mistakes), model accuracy suffers. We'll inject symmetric noise at different rates and observe the degradation.

In [ ]:
# Generate make_moons dataset
X, y = make_moons(n_samples=500, noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Inject symmetric noise at different rates
noise_rates = [0.0, 0.05, 0.10, 0.15, 0.20]
accuracies = []

for noise_rate in noise_rates:
    y_noisy = y_train.copy()
    n_flip = int(len(y_noisy) * noise_rate)
    flip_indices = np.random.choice(len(y_noisy), n_flip, replace=False)
    y_noisy[flip_indices] = 1 - y_noisy[flip_indices]
    
    # Train model
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_noisy)
    acc = accuracy_score(y_test, model.predict(X_test))
    accuracies.append(acc)
    print(f"Noise {noise_rate:.0%}: test accuracy = {acc:.3f}")

# Plot
plt.figure(figsize=(8, 5))
plt.plot([r*100 for r in noise_rates], accuracies, 'o-', linewidth=2, markersize=8, color='steelblue')
plt.xlabel('Label Noise Rate (%)', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Impact of Symmetric Label Noise on Logistic Regression', fontsize=13)
plt.grid(True, alpha=0.3)
plt.ylim([0.5, 1.0])
plt.tight_layout()
plt.show()

## Part 2: Class Imbalance

When one class appears much more often than another, standard accuracy hides poor minority-class performance. We'll create an imbalanced dataset and compare naive training with class-weighted training.

In [ ]:
# Create imbalanced dataset (1:10 ratio)
X_imb, y_imb = make_moons(n_samples=500, noise=0.1, random_state=42)
imbalance_mask = np.concatenate([np.arange(100), np.where(y_imb == 0)[0][100:]])
X_imb, y_imb = X_imb[imbalance_mask], y_imb[imbalance_mask]
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42
)

print(f"Class distribution in training set: {np.bincount(y_train_imb)}")
print()

# Train without class weights
model_unweighted = LogisticRegression(random_state=42, max_iter=1000)
model_unweighted.fit(X_train_imb, y_train_imb)
y_pred_unweighted = model_unweighted.predict(X_test_imb)

# Train with class weights
model_weighted = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model_weighted.fit(X_train_imb, y_train_imb)
y_pred_weighted = model_weighted.predict(X_test_imb)

# Compare metrics
print("Without class weights:")
print(f"  Accuracy:  {accuracy_score(y_test_imb, y_pred_unweighted):.3f}")
print(f"  Precision: {precision_score(y_test_imb, y_pred_unweighted):.3f}")
print(f"  Recall:    {recall_score(y_test_imb, y_pred_unweighted):.3f}")
print(f"  F1 Score:  {f1_score(y_test_imb, y_pred_unweighted):.3f}")
print()

print("With class_weight='balanced':")
print(f"  Accuracy:  {accuracy_score(y_test_imb, y_pred_weighted):.3f}")
print(f"  Precision: {precision_score(y_test_imb, y_pred_weighted):.3f}")
print(f"  Recall:    {recall_score(y_test_imb, y_pred_weighted):.3f}")
print(f"  F1 Score:  {f1_score(y_test_imb, y_pred_weighted):.3f}")

## Part 3: Data Augmentation Validity

Augmentation (adding noise to features) can improve generalization. We'll train two models—one on clean data, one on augmented data—and compare their decision boundaries.

In [ ]:
# Generate clean dataset
X_aug, y_aug = make_moons(n_samples=300, noise=0.05, random_state=42)
X_train_aug, X_test_aug, y_train_aug, y_test_aug = train_test_split(
    X_aug, y_aug, test_size=0.3, random_state=42
)

# Train on clean data
model_clean = LogisticRegression(random_state=42, max_iter=1000)
model_clean.fit(X_train_aug, y_train_aug)
acc_clean = accuracy_score(y_test_aug, model_clean.predict(X_test_aug))

# Augment training data with Gaussian noise
X_train_aug_noisy = X_train_aug + np.random.randn(*X_train_aug.shape) * 0.15
model_augmented = LogisticRegression(random_state=42, max_iter=1000)
model_augmented.fit(X_train_aug_noisy, y_train_aug)
acc_aug = accuracy_score(y_test_aug, model_augmented.predict(X_test_aug))

print(f"Clean training accuracy:      {acc_clean:.3f}")
print(f"Augmented training accuracy:  {acc_aug:.3f}")
print()

# Plot decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Create mesh
x_min, x_max = X_aug[:, 0].min() - 0.5, X_aug[:, 0].max() + 0.5
y_min, y_max = X_aug[:, 1].min() - 0.5, X_aug[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
mesh_points = np.c_[xx.ravel(), yy.ravel()]

# Plot clean model
Z_clean = model_clean.predict_proba(mesh_points)[:, 1].reshape(xx.shape)
axes[0].contourf(xx, yy, Z_clean, levels=20, cmap='RdYlBu')
axes[0].scatter(X_train_aug[y_train_aug == 0, 0], X_train_aug[y_train_aug == 0, 1], 
               c='red', marker='o', alpha=0.5, s=20)
axes[0].scatter(X_train_aug[y_train_aug == 1, 0], X_train_aug[y_train_aug == 1, 1], 
               c='blue', marker='o', alpha=0.5, s=20)
axes[0].set_title(f'Clean Training (acc={acc_clean:.3f})', fontsize=12)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Plot augmented model
Z_aug = model_augmented.predict_proba(mesh_points)[:, 1].reshape(xx.shape)
axes[1].contourf(xx, yy, Z_aug, levels=20, cmap='RdYlBu')
axes[1].scatter(X_train_aug[y_train_aug == 0, 0], X_train_aug[y_train_aug == 0, 1], 
               c='red', marker='o', alpha=0.5, s=20)
axes[1].scatter(X_train_aug[y_train_aug == 1, 0], X_train_aug[y_train_aug == 1, 1], 
               c='blue', marker='o', alpha=0.5, s=20)
axes[1].set_title(f'Augmented Training (acc={acc_aug:.3f})', fontsize=12)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

## Part 4: Active Learning Simulation

Active learning queries the most uncertain unlabeled points, reducing the number of labels needed. We compare uncertainty sampling (active) vs. random sampling.

In [ ]:
# Generate dataset and split into labeled and pool
X_al, y_al = make_moons(n_samples=500, noise=0.1, random_state=42)

# Start with 10 labeled points and a pool of 400 unlabeled
initial_indices = np.random.choice(len(X_al), 10, replace=False)
pool_indices = np.array([i for i in range(len(X_al)) if i not in initial_indices])

X_labeled = X_al[initial_indices]
y_labeled = y_al[initial_indices]
X_pool = X_al[pool_indices]
y_pool = y_al[pool_indices]

# Test set (fixed)
_, X_test_al, _, y_test_al = train_test_split(X_al, y_al, test_size=0.2, random_state=42)

# Uncertainty sampling
accuracies_active = []
pool_copy = pool_indices.copy()
X_labeled_active = X_labeled.copy()
y_labeled_active = y_labeled.copy()

for i in range(30):
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_labeled_active, y_labeled_active)
    acc = accuracy_score(y_test_al, model.predict(X_test_al))
    accuracies_active.append(acc)
    
    # Query most uncertain point
    probs = model.predict_proba(X_pool)
    uncertainty = np.abs(probs[:, 1] - 0.5)
    query_idx = np.argmin(uncertainty)
    
    # Add to labeled set
    X_labeled_active = np.vstack([X_labeled_active, X_pool[query_idx]])
    y_labeled_active = np.append(y_labeled_active, y_pool[query_idx])
    X_pool = np.delete(X_pool, query_idx, axis=0)

# Random sampling
accuracies_random = []
X_labeled_random = X_labeled.copy()
y_labeled_random = y_labeled.copy()
X_pool_random = X_pool.copy()

for i in range(30):
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_labeled_random, y_labeled_random)
    acc = accuracy_score(y_test_al, model.predict(X_test_al))
    accuracies_random.append(acc)
    
    # Query random point
    query_idx = np.random.choice(len(X_pool_random))
    X_labeled_random = np.vstack([X_labeled_random, X_pool_random[query_idx]])
    y_labeled_random = np.append(y_labeled_random, y_pool[query_idx])
    X_pool_random = np.delete(X_pool_random, query_idx, axis=0)

# Plot learning curves
plt.figure(figsize=(9, 5))
queries = np.arange(10, 10 + len(accuracies_active))
plt.plot(queries, accuracies_active, 'o-', linewidth=2, markersize=6, 
         label='Uncertainty Sampling (Active)', color='steelblue')
plt.plot(queries, accuracies_random, 's-', linewidth=2, markersize=6, 
         label='Random Sampling', color='coral')
plt.xlabel('Number of Labeled Points', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Active Learning vs. Random Sampling', fontsize=13)
plt.legend(fontsize=11, loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## What to Notice

**Label Noise:**
- Even 5–10% label noise significantly reduces accuracy
- Noise at 20% can reduce accuracy by 10–15 percentage points
- Robust annotation protocols and multi-annotator review are essential

**Class Imbalance:**
- Accuracy alone hides poor minority-class performance
- Precision, recall, and F1 score reveal class-specific behavior
- Class weights automatically adjust the learning objective to penalize minority-class errors more

**Data Augmentation:**
- Augmentation can improve generalization by exposing the model to realistic variations
- However, augmentation also smooths decision boundaries, which may hurt tight classifiers
- The effect depends on augmentation magnitude and the problem's inherent complexity

**Active Learning:**
- Uncertainty sampling queries harder examples, leading to faster learning
- Active learning requires 2–3× fewer labeled examples to reach the same accuracy as random sampling
- The benefit grows as the pool becomes larger and unlabeled data is abundant